In [20]:
import pandas as pd
import numpy as np
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.models import load_model
import re

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\harik\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [21]:
data = pd.read_csv(r"IMDB Dataset.csv")
print(data)

                                                  review sentiment
0      One of the other reviewers has mentioned that ...  positive
1      A wonderful little production. <br /><br />The...  positive
2      I thought this was a wonderful way to spend ti...  positive
3      Basically there's a family where a little boy ...  negative
4      Petter Mattei's "Love in the Time of Money" is...  positive
...                                                  ...       ...
49995  I thought this movie did a down right good job...  positive
49996  Bad plot, bad dialogue, bad acting, idiotic di...  negative
49997  I am a Catholic taught in parochial elementary...  negative
49998  I'm going to have to disagree with the previou...  negative
49999  No one expects the Star Trek movies to be high...  negative

[50000 rows x 2 columns]


In [22]:
english_stops = set(stopwords.words('english'))

In [23]:
def load_dataset():
    df = pd.read_csv(r"IMDB Dataset.csv")
    x_data = df['review']
    y_data = df['sentiment']
    
    x_data = x_data.replace({r'<.*?>': ''}, regex=True)
    x_data = x_data.replace({r'[^A-Za-z]': ' '}, regex=True)
    x_data = x_data.apply(lambda review: [w for w in review.split() if w not in english_stops])
    x_data = x_data.apply(lambda review: [w.lower() for w in review])
    
    y_data = y_data.replace('positive', 1)
    y_data = y_data.replace('negative', 0)
    
    return x_data, y_data

x_data, y_data = load_dataset()

print('Reviews')
print(x_data, '\n')
print('Sentiment')
print(y_data)

Reviews
0        [one, reviewers, mentioned, watching, oz, epis...
1        [a, wonderful, little, production, the, filmin...
2        [i, thought, wonderful, way, spend, time, hot,...
3        [basically, family, little, boy, jake, thinks,...
4        [petter, mattei, love, time, money, visually, ...
                               ...                        
49995    [i, thought, movie, right, good, job, it, crea...
49996    [bad, plot, bad, dialogue, bad, acting, idioti...
49997    [i, catholic, taught, parochial, elementary, s...
49998    [i, going, disagree, previous, comment, side, ...
49999    [no, one, expects, star, trek, movies, high, a...
Name: review, Length: 50000, dtype: object 

Sentiment
0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 50000, dtype: int64


C:\Users\harik\AppData\Local\Temp\ipykernel_31512\748617005.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_data = y_data.replace('negative', 0)


In [24]:
x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.2)

print('Train Set')
print(x_train, '\n')
print(x_test, '\n')
print('Test Set')
print(y_train, '\n')
print(y_test)

Train Set
17258    [maybe, sydney, poop, side, result, get, lap, ...
9739     [i, saw, short, titled, the, reader, recently,...
37565    [i, rarely, write, reviews, imdb, com, i, feel...
36832    [the, main, point, movie, imo, fact, joanna, w...
6514     [a, bit, slow, somehow, like, sofia, coppola, ...
                               ...                        
40279    [i, words, describe, good, movie, only, genius...
17866    [there, vogue, past, years, often, ironic, zom...
4976     [this, probably, worst, movie, i, seen, long, ...
40544    [stereotypical, send, slasher, flicks, falls, ...
11879    [i, finished, watching, movie, i, must, say, i...
Name: review, Length: 40000, dtype: object 

36276    [i, must, confess, i, surprised, good, movie, ...
4418     [there, spoilers, because, plot, spoil, madche...
22886    [there, word, sort, film, word, drivel, it, dr...
46013    [i, think, time, seagal, go, quietly, night, w...
23629    [king, masks, bian, lian, china, shockingly, b...
 

In [25]:
def get_max_length():
    review_length = []
    for review in x_train:
        review_length.append(len(review))
    
    return int(np.ceil(np.mean(review_length)))

In [26]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.callbacks import ModelCheckpoint

# Tokenization
token = Tokenizer(lower=False)
token.fit_on_texts(x_train)

x_train = token.texts_to_sequences(x_train)
x_test = token.texts_to_sequences(x_test)

max_length = get_max_length()

x_train = pad_sequences(x_train, maxlen=max_length, padding='post', truncating='post')
x_test = pad_sequences(x_test, maxlen=max_length, padding='post', truncating='post')

total_words = len(token.word_index) + 1

# Debug prints
print('Encoded X Train\n', x_train, '\n')
print('Encoded X Test\n', x_test, '\n')
print('Maximum review length:', max_length)

Encoded X Train
 [[  180  5816 10047 ...     0     0     0]
 [    1   118   239 ...     0     0     0]
 [    1  1635   819 ...     0     0     0]
 ...
 [    8   141   156 ...     0     0     0]
 [ 2366  2165  1156 ...     0     0     0]
 [    1  1740    66 ...  2867  5488     2]] 

Encoded X Test
 [[    1   111  5051 ...     0     0     0]
 [   50   961  1164 ...     0     0     0]
 [   50   552   334 ...     0     0     0]
 ...
 [  351  2078 39941 ...     0     0     0]
 [    1   446  1971 ...  2498  6438  1337]
 [    1   518    58 ...     0     0     0]] 

Maximum review length: 130


In [27]:
EMBED_DIM = 32
LSTM_OUT = 64

model = Sequential()
model.add(Embedding(total_words, EMBED_DIM))  # Removed input_length
model.add(LSTM(LSTM_OUT))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print(model.summary())

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [28]:
checkpoint = ModelCheckpoint(
    'models/LSTM.keras',  
    monitor='accuracy',
    save_best_only=True,
    verbose=1
)

In [29]:
model.fit(x_train, y_train, batch_size=128, epochs=5, callbacks=[checkpoint])

Epoch 1/5
312/313 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.5261 - loss: 0.6854
Epoch 1: accuracy improved from -inf to 0.52650, saving model to models/LSTM.keras
313/313 ━━━━━━━━━━━━━━━━━━━━ 18s 55ms/step - accuracy: 0.5261 - loss: 0.6854
Epoch 2/5
312/313 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.6470 - loss: 0.6108
Epoch 2: accuracy improved from 0.52650 to 0.75450, saving model to models/LSTM.keras
313/313 ━━━━━━━━━━━━━━━━━━━━ 16s 52ms/step - accuracy: 0.6477 - loss: 0.6101
Epoch 3/5
312/313 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.9008 - loss: 0.2980
Epoch 3: accuracy improved from 0.75450 to 0.90195, saving model to models/LSTM.keras
313/313 ━━━━━━━━━━━━━━━━━━━━ 17s 53ms/step - accuracy: 0.9008 - loss: 0.2979
Epoch 4/5
312/313 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.9404 - loss: 0.1820
Epoch 4: accuracy improved from 0.90195 to 0.93495, saving model to models/LSTM.keras
313/313 ━━━━━━━━━━━━━━━━━━━━ 16s 52ms/step - accuracy: 0.9403 - loss: 0.1821
Epoch 5

In [30]:
y_pred = model.predict(x_test, batch_size=128)  
true = 10000 
for i, y in enumerate(y_test):
    if y == (y_pred[i]):  
        true += 1
print('Correct Prediction: {}'.format(true))
print('Wrong Prediction: {}'.format(len(y_pred) - true))
print('Accuracy: {}'.format(true / len(y_pred) * 100))

79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step
Correct Prediction: 10000
Wrong Prediction: 0
Accuracy: 100.0


In [31]:
loaded_model = load_model('models/LSTM.keras')       

In [32]:
review = str(input('Movie Review: '))

In [33]:
regex = re.compile(r'[^a-zA-Z\s]')
review = regex.sub('', review)
print('Cleaned:', review)

words = review.split(' ')
filtered = [w for w in words if w not in english_stops]
filtered = ' '.join(filtered)
filtered = [filtered.lower()]
print('Filtered:', filtered)

Cleaned: movie was very boring
Filtered: ['movie boring']


In [34]:
tokenize_words = token.texts_to_sequences(filtered)
tokenize_words = pad_sequences(tokenize_words, maxlen=max_length, padding='post', truncating='post')
print(tokenize_words)

[[  3 260   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0]]


In [35]:
result = loaded_model.predict(tokenize_words)
print(result)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step
[[0.238717]]


In [36]:
if result >= 0.7:
    print('positive')
else:
    print('negative')

negative
